# GoogLeNet（CIFAR100を用いた画像認識）

---
## 目的
畳み込みニューラルネットワークのモデルとして，GoogLeNet（Inception v1）を用いてCIFAR100データセットの100クラス物体認識を行い，複数サイズの畳み込みを並列に組み合わせるInceptionモジュールの構造とその利点について理解する．

## モジュールのインポート
はじめに必要なモジュールをインポートしたのち，GPUを使用した計算が可能かどうかを確認する．

In [ ]:
from time import time

import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import torchsummary

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

## データセットの読み込みと確認
CIFAR100データセットを読み込みます．CIFAR100はCIFAR10と同じく32×32ピクセルのカラー画像データセットですが，クラス数が100（1クラスあたり学習用500枚，評価用100枚）とより細かいクラス分けになっています．

学習用データに対しては，`RandomCrop`と`RandomHorizontalFlip`によるData Augmentationを適用します．

In [ ]:
transform_train = transforms.Compose([transforms.RandomCrop(32, padding=4),
                                       transforms.RandomHorizontalFlip(),
                                       transforms.ToTensor()])
transform_test = transforms.Compose([transforms.ToTensor()])

train_data = torchvision.datasets.CIFAR100(root="./", train=True, transform=transform_train, download=True)
test_data = torchvision.datasets.CIFAR100(root="./", train=False, transform=transform_test, download=True)

print(len(train_data), len(test_data))

## Inceptionモジュール
GoogLeNetは，2014年のILSVRCで優勝したモデルであり，「Inceptionモジュール」と呼ばれる構造を積み重ねることで，パラメータ数を抑えながら高い認識精度を達成したことで知られています．

通常の畳み込み層では，1つの層に対して1つのカーネルサイズしか使用しませんが，画像中に写る物体の大きさは様々であるため，どのカーネルサイズが適切かは一概には決まりません．Inceptionモジュールでは，1×1，3×3，5×5の畳み込みとMaxPoolingを並列に配置し，それぞれの出力をチャンネル方向に連結（concatenate）することで，複数のスケールの特徴を同時に捉えられるようにしています．

### 1×1畳み込みによる次元削減
3×3や5×5の畳み込みをそのまま並列に配置すると，チャンネル数の増加に伴って計算量が急激に増大してしまいます．そこで，3×3・5×5畳み込みの直前と，MaxPoolingの直後に1×1畳み込みを挿入し，チャンネル数を一度削減（ボトルネック化）してから畳み込みを行うことで，計算量を大幅に削減しています．

In [ ]:
class InceptionModule(nn.Module):
    def __init__(self, in_channels, ch1x1, ch3x3red, ch3x3, ch5x5red, ch5x5, pool_proj):
        super().__init__()
        self.branch1 = nn.Sequential(
            nn.Conv2d(in_channels, ch1x1, kernel_size=1, bias=False),
            nn.BatchNorm2d(ch1x1),
            nn.ReLU(inplace=True),
        )
        self.branch2 = nn.Sequential(
            nn.Conv2d(in_channels, ch3x3red, kernel_size=1, bias=False),
            nn.BatchNorm2d(ch3x3red),
            nn.ReLU(inplace=True),
            nn.Conv2d(ch3x3red, ch3x3, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(ch3x3),
            nn.ReLU(inplace=True),
        )
        self.branch3 = nn.Sequential(
            nn.Conv2d(in_channels, ch5x5red, kernel_size=1, bias=False),
            nn.BatchNorm2d(ch5x5red),
            nn.ReLU(inplace=True),
            nn.Conv2d(ch5x5red, ch5x5, kernel_size=5, padding=2, bias=False),
            nn.BatchNorm2d(ch5x5),
            nn.ReLU(inplace=True),
        )
        self.branch4 = nn.Sequential(
            nn.MaxPool2d(kernel_size=3, stride=1, padding=1),
            nn.Conv2d(in_channels, pool_proj, kernel_size=1, bias=False),
            nn.BatchNorm2d(pool_proj),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        # 4つのブランチの出力をチャンネル方向に連結する
        return torch.cat([self.branch1(x), self.branch2(x), self.branch3(x), self.branch4(x)], dim=1)

### GoogLeNetの定義
Inceptionモジュールを複数積み重ねてGoogLeNet全体を構築します．CIFAR100（32×32入力）に合わせて，最初の畳み込み（stem）は3×3畳み込み1層のみとし，Inceptionモジュールの間には特徴マップサイズを半分にするMaxPoolingを2箇所のみ挿入します．各Inceptionモジュールの出力チャンネル数（`ch1x1`, `ch3x3red`, `ch3x3`, `ch5x5red`, `ch5x5`, `pool_proj`）の割り当ては，元論文のTable 1に記載された値をそのまま用いています．

また，元論文では出力層の直前に大きな全結合層を用いる代わりにGlobal Average Poolingを使用しており，これによりパラメータ数を大きく削減しています．本実装でもこの点を踏襲します．

In [ ]:
class GoogLeNet(nn.Module):
    def __init__(self, n_class=100):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, 192, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(192),
            nn.ReLU(inplace=True),
        )

        self.inception3a = InceptionModule(192, 64, 96, 128, 16, 32, 32)     # out: 256
        self.inception3b = InceptionModule(256, 128, 128, 192, 32, 96, 64)   # out: 480
        self.maxpool1 = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)

        self.inception4a = InceptionModule(480, 192, 96, 208, 16, 48, 64)    # out: 512
        self.inception4b = InceptionModule(512, 160, 112, 224, 24, 64, 64)   # out: 512
        self.inception4c = InceptionModule(512, 128, 128, 256, 24, 64, 64)   # out: 512
        self.inception4d = InceptionModule(512, 112, 144, 288, 32, 64, 64)   # out: 528
        self.inception4e = InceptionModule(528, 256, 160, 320, 32, 128, 128) # out: 832
        self.maxpool2 = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)

        self.inception5a = InceptionModule(832, 256, 160, 320, 32, 128, 128) # out: 832
        self.inception5b = InceptionModule(832, 384, 192, 384, 48, 128, 128) # out: 1024

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.dropout = nn.Dropout(0.4)
        self.fc = nn.Linear(1024, n_class)

    def forward(self, x):
        x = self.stem(x)

        x = self.inception3a(x)
        x = self.inception3b(x)
        x = self.maxpool1(x)

        x = self.inception4a(x)
        x = self.inception4b(x)
        x = self.inception4c(x)
        x = self.inception4d(x)
        x = self.inception4e(x)
        x = self.maxpool2(x)

        x = self.inception5a(x)
        x = self.inception5b(x)

        x = self.avgpool(x)
        x = x.view(x.size(0), -1)
        x = self.dropout(x)
        x = self.fc(x)
        return x

## ネットワークの作成
定義したGoogLeNetを作成し，`.to(device)`によりモデルのパラメータを`device`上に配置します．

最適化手法にはモーメンタムSGDを用い，学習率0.01，モーメンタム0.9，weight decayを5e-4とします．最後に`torchsummary.summary()`でネットワークの詳細情報を表示します．

In [ ]:
model = GoogLeNet(n_class=100)
model = model.to(device)

optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9, weight_decay=5e-4)

torchsummary.summary(model, (3, 32, 32), device=device.type)

## 学習
読み込んだCIFAR100データセットと作成したネットワークを用いて学習を行います．ミニバッチサイズを128，学習エポック数を20とします．

In [ ]:
batch_size = 128
epoch_num = 20

train_loader = torch.utils.data.DataLoader(train_data, batch_size=batch_size, shuffle=True, num_workers=2)

criterion = nn.CrossEntropyLoss()
criterion = criterion.to(device)

model.train()

train_start = time()
for epoch in range(1, epoch_num + 1):
    sum_loss = 0.0
    count = 0

    for image, label in train_loader:
        image = image.to(device)
        label = label.to(device)

        y = model(image)

        loss = criterion(y, label)
        model.zero_grad()
        loss.backward()
        optimizer.step()

        sum_loss += loss.item()
        pred = torch.argmax(y, dim=1)
        count += torch.sum(pred == label)

    print("epoch: {}, mean loss: {}, mean accuracy: {}, elapsed time: {}".format(
        epoch, sum_loss / len(train_loader), count.item() / len(train_data), time() - train_start))

## テスト
学習したネットワークモデルを用いて評価を行います．

In [ ]:
test_loader = torch.utils.data.DataLoader(test_data, batch_size=100, shuffle=False)

model.eval()

count = 0
with torch.no_grad():
    for image, label in test_loader:
        image = image.to(device)
        label = label.to(device)

        y = model(image)

        pred = torch.argmax(y, dim=1)
        count += torch.sum(pred == label)

print("test accuracy: {}".format(count.item() / len(test_data)))

## ImageNet版GoogLeNetとCIFAR版（本ノートブック）の違い
このノートブックで実装したGoogLeNetは，CIFAR100（32×32入力）向けに元論文（ImageNet, 224×224入力）の構造を簡略化したものです．主な違いは次の通りです．

| | ImageNet版 | CIFAR版（本ノートブック） |
|---|---|---|
| 入力画像サイズ | 224×224 | 32×32 |
| stem（最初の畳み込み部） | 7×7 conv (stride2) → 3×3 maxpool (stride2) → 1×1/3×3 conv → 3×3 maxpool (stride2) の4層 | 3×3 conv (stride1) の1層のみ |
| 全体のダウンサンプリング回数 | stemで3回 + Inceptionモジュール間で2回の計5回 | Inceptionモジュール間の2回のみ |
| 補助分類器（Auxiliary Classifier） | 学習中段に2箇所挿入し，勾配消失を緩和 | なし（実装の簡略化のため省略） |
| Batch Normalization | なし（2014年の原論文にはBNは存在しない） | あり（各畳み込み後に付加し，学習を安定化） |
| 出力層のクラス数 | 1000 | 100（CIFAR100） |

とくにstemの簡略化と補助分類器の省略は，元論文からの大きな変更点です．224×224入力を前提とした7×7 stride2畳み込みや3回のダウンサンプリングを32×32画像にそのまま適用すると，Inceptionモジュールに到達する前に特徴マップが小さくなりすぎてしまうため，stemを3×3 stride1の畳み込み1層のみに簡略化し，早期のダウンサンプリングを避けています．また，補助分類器は学習の安定化に寄与しますが，実装が複雑になるため，このノートブックでは省略しています（本来はネットワーク中段のInceptionモジュールの出力から分岐する小さな分類ヘッドを追加し，メインの損失と重み付けして合算します）．

## 課題

1. Inceptionモジュール内の各ブランチ（1×1，3×3，5×5，pooling projection）のチャンネル数の割り当てを変更し，パラメータ数と認識精度がどのように変化するか確認しましょう．

2. Inceptionモジュールの段数（積み重ねる数）やMaxPoolingを挿入する位置を変更し，ネットワークの深さ・ダウンサンプリングのタイミングが認識精度に与える影響を確認しましょう．

3. 学習の設定（ミニバッチサイズ，学習率，Epoch数，最適化手法など）を変更し，認識精度の変化を確認しましょう．